# 🧬 Master Pipeline Project 5
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup
Eseguire per configurare Drive e caricare le funzioni Helper.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b osagie5 https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data

In [ ]:
# ── Setup Logica Demo (Stage 4) ─────────────────────────────────
import torch, os, gradio as gr
from PIL import Image, ImageDraw
import torchvision.transforms as T
from models.extractor import DINOv2Extractor
from models.lora import apply_lora_to_dinov2
from models.correspondence import SemanticCorrespondenceModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

def launch_stage_demo(title, ckpt_name=None, layer_idx=-1, show_layer_slider=False):
    def get_prediction(src_img, trg_img, x, y, current_layer):
        # Creamo un estrattore al volo per il layer scelto
        backbone = DINOv2Extractor(model_name='dinov2_vitb14', layer=current_layer, freeze=True)
        if ckpt_name:
            path = f'/content/drive/MyDrive/AML/Checkpoints/{ckpt_name}/best.pth'
            if os.path.exists(path):
                ckpt = torch.load(path, map_location=device)
                backbone.model = apply_lora_to_dinov2(backbone.model, rank=ckpt['args'].get('lora_rank', 16))
                model = SemanticCorrespondenceModel(backbone=backbone, use_adaptive_win=True).to(device)
                model.load_state_dict(ckpt['model_state_dict'])
            else: model = SemanticCorrespondenceModel(backbone=backbone, use_adaptive_win=True).to(device)
        else:
            model = SemanticCorrespondenceModel(backbone=backbone, use_adaptive_win=True).to(device)
        
        model.eval()
        s_t, t_t = transform(src_img).unsqueeze(0).to(device), transform(trg_img).unsqueeze(0).to(device)
        src_kp = torch.tensor([[[x*(224/src_img.width), y*(224/src_img.height)]]], device=device).float()
        with torch.no_grad():
            out = model(s_t, t_t, src_kps=src_kp)
        pkp = out['pred_kps'][0,0].cpu().numpy()
        return pkp[0]*(trg_img.width/224), pkp[1]*(trg_img.height/224)

    def gradio_fn(s_img, t_img, lyr, evt: gr.SelectData):
        tx, ty = get_prediction(s_img, t_img, evt.index[0], evt.index[1], int(lyr))
        s_o, t_o = s_img.copy(), t_img.copy()
        ImageDraw.Draw(s_o).ellipse([evt.index[0]-6, evt.index[1]-6, evt.index[0]+6, evt.index[1]+6], fill='red', outline='white')
        ImageDraw.Draw(t_o).ellipse([tx-6, ty-6, tx+6, ty+6], fill='green', outline='white')
        return s_o, t_o

    with gr.Blocks() as d:
        gr.Markdown(f"### {title}")
        with gr.Row():
            lyr_ctrl = gr.Slider(0, 11, value=layer_idx if layer_idx != -1 else 11, step=1, label='Transformer Layer (0-11)', visible=show_layer_slider)
        with gr.Row():
            si = gr.Image(type='pil', label='Source (Clicca)')
            ti = gr.Image(type='pil', label='Target Image')
        with gr.Row():
            so, to = gr.Image(type='pil', label='Selection'), gr.Image(type='pil', label='Prediction')
        si.select(gradio_fn, [si, ti, lyr_ctrl], [so, to])
    d.launch(share=True, inline=False)

## 🔍 1. Stage 1: Backbone Analysis
Valutazioni iniziali di DINOv2 vergine.

In [ ]:
# 1.1 Analisi PCK Baseline (Layer 11)
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --results_file /content/drive/MyDrive/AML/Results/baseline.txt

In [ ]:
# 🎮 Demo Baseline (Fixed Layer 11)
launch_stage_demo('DINOv2 Baseline (Layer 11)', ckpt_name=None, layer_idx=11)

In [ ]:
# 1.2 Analisi PCK Layer-wise
for layer in [4, 8, 11]:
    print(f'\n--- Testing Layer {layer} ---')
    !python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --layer {layer} --results_file /content/drive/MyDrive/AML/Results/layer_{layer}.txt

In [ ]:
# 🎮 Demo Layer-wise Explorer
# Usa lo slider per vedere come cambia la precisione a diverse profondità!
launch_stage_demo('DINOv2 Layer-wise Explorer', ckpt_name=None, show_layer_slider=True)

## 🚀 2. Stage 2: Fine-Tuning Efficiente (LoRA)
Addestramento modulare.

In [ ]:
# 2.1 Solo LoRA (Senza Curriculum)
!python train.py --epochs 5 --curriculum_epochs 0 --num_workers 4 --output_dir ./checkpoints/lora_only --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_only

In [ ]:
# 🎮 Demo LoRA Only
launch_stage_demo('LoRA Only (Inference)', ckpt_name='lora_only')

In [ ]:
# 2.2 LoRA + Curriculum (Final Model)
!python train.py --epochs 5 --curriculum_epochs 2 --num_workers 4 --output_dir ./checkpoints/lora_curriculum --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_curriculum

## 🎯 3. Stage 3: Raffinamento Adattivo
Ablation dell'Adaptive Windowing.

In [ ]:
CKPT = '/content/drive/MyDrive/AML/Checkpoints/lora_curriculum/best.pth'
!python evaluate.py --checkpoint {CKPT} --results_file /content/drive/MyDrive/AML/Results/s3_with.txt
!python evaluate.py --checkpoint {CKPT} --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/s3_without.txt

## 🌟 4. Stage 4: Valutazione ed Estensioni
Demo finale del miglior modello.

In [ ]:
# 🎮 Demo Finale (LoRA + Curriculum)
launch_stage_demo('Modello Finale (LoRA + Curriculum)', ckpt_name='lora_curriculum')